# Date Table for Data Warehouse

This notebook creates a comprehensive **date dimension table** for the data warehouse.

## Purpose
A date table (date dimension) is a central reference table in a data warehouse, used for time intelligence calculations and time-based analyses in Power BI and other BI tools.

## What does this notebook do?

1. **Defines a date range** (1900-2222)
2. **Creates the schema `dim`** in the data warehouse
3. **Generates a base table** with all date values in the defined range
4. **Extends the table** with calculated columns for:
   - Calendar hierarchies (year, month, quarter, week)
   - Time-based offsets (difference from today)
   - Weekdays and workdays
   - Helper fields for sorting and formatting

## Target Location
- **Warehouse**: `Date`
- **Schema**: `dim`
- **Tables**: `DateRange` (base) and `DateTable` (complete)

## 1. Define Date Range

In [1]:
# Configuration: date range of the table
# Format: dd.mm.yyyy (German format, parsed in the notebook)
start_date = "01.01.1900"
end_date   = "31.12.2222"

## 2. Create Schema
Creates the `dim` (dimension) schema in the data warehouse if it does not already exist.

In [2]:
%%tsql -artifact Date -type Warehouse
-- This script ensures that the 'dim' schema exists in the warehouse.
-- 1. Checks sys.schemas to see whether a schema named 'dim' already exists.
-- 2. If it does not exist, the 'dim' schema is created via a dynamic EXEC command.

IF NOT EXISTS (SELECT * FROM sys.schemas WHERE name = 'dim')
    EXEC('CREATE SCHEMA dim;')

## 3. Generate Date Table and Write to Warehouse

This cell:
- Parses the date values from the German format (dd.mm.yyyy)
- Builds a list of all days between the start and end date
- Converts these into a Spark DataFrame
- Writes the data directly into the `Date` data warehouse in the `dim` schema

In [3]:
from datetime import datetime
import notebookutils

# 3. Generate the date table without Spark and write it to the warehouse
# Uses T-SQL inside the warehouse (instead of Spark) to build the dim.DateRange table.

# Parse the date values from the German format (dd.mm.yyyy) into Python date objects
start_dt = datetime.strptime(start_date, "%d.%m.%Y").date()
end_dt = datetime.strptime(end_date, "%d.%m.%Y").date()

# ISO format (YYYY-MM-DD) for T-SQL
start_iso = start_dt.isoformat()
end_iso = end_dt.isoformat()

# Open a connection to the "Date" warehouse
conn = notebookutils.data.connect_to_artifact("Date", None, "Warehouse")

# T-SQL script that creates/truncates and populates the dim.DateRange table
# Note: PRIMARY KEY is NOT used because it is not supported in this edition
sql = f"""
DECLARE @StartDate date = '{start_iso}';
DECLARE @EndDate   date = '{end_iso}';

IF OBJECT_ID('dim.DateRange', 'U') IS NULL
BEGIN
    CREATE TABLE dim.DateRange
    (
        DateRange date NOT NULL
    );
END
ELSE
BEGIN
    TRUNCATE TABLE dim.DateRange;
END;

WITH Dates AS
(
    SELECT DATEADD(DAY, value, @StartDate) AS d
    FROM GENERATE_SERIES(0, DATEDIFF(DAY, @StartDate, @EndDate), 1)
)
INSERT INTO dim.DateRange (DateRange)
SELECT d
FROM Dates;
"""

# Execute the script in the warehouse
conn.query(sql)

## 4. Create the Extended Date Table with All Attributes

This SparkSQL query:
- Reads the base date table `DateRange`
- Calculates all additional columns for analytics:
  - **Calendar hierarchies**: year, month, quarter, calendar week
  - **Offsets**: differences from today (days, months, years)
  - **Workdays**: flags for weekends, workdays, and business days
  - **Sort helper fields**: for correct sorting in Power BI
- Writes the result to the final table `DateTable`

The table can then be used in Power BI as a date dimension for time intelligence.

In [4]:
%%tsql -artifact Date -type Warehouse
SET DATEFIRST 7;  -- Lock to US default (Sun=1, Mon=2, ..., Sat=7) — matches the BETWEEN ranges below

IF OBJECT_ID('dim.DateTable', 'U') IS NOT NULL
BEGIN
    DROP TABLE dim.DateTable;
END;

DECLARE @Today date = CAST(SYSDATETIME() AS date);
DECLARE @BaseDate date = DATEFROMPARTS(1900, 1, 1);

CREATE TABLE dim.DateTable
AS
SELECT 
    CAST(D.DateRange AS DATE) AS [Date]  -- Primary key: the date
    
    -- === Calendar hierarchies ===
    , DATEFROMPARTS(YEAR(D.DateRange), MONTH(D.DateRange), 1) AS CY_Month  -- First day of the month
    , DATEFROMPARTS(YEAR(D.DateRange), 1, 1) AS CY_Year                    -- First day of the year
    
    -- === Months ===
    -- Uses 1900 as base year so sorting works independently of the actual year
    , DATEFROMPARTS(1900, MONTH(D.DateRange), 1) AS Months_Long  -- For long month names
    , DATEFROMPARTS(1900, MONTH(D.DateRange), 1) AS Months       -- For short month names
    
    -- === Quarters ===
    , 'Q' + CAST(DATEPART(QUARTER, D.DateRange) AS VARCHAR(1)) 
      + '/' + CAST(YEAR(D.DateRange) AS VARCHAR(4)) AS QuarterYear          -- "Q1/2024"
    , 'Q' + CAST(DATEPART(QUARTER, D.DateRange) AS VARCHAR(1)) AS Quarters  -- "Q1"
    , DATEADD(QUARTER, DATEDIFF(QUARTER, @BaseDate, D.DateRange), @BaseDate) AS QuarterYearSort  -- First day of the quarter
    , DATEFROMPARTS(
          1900, 
          MONTH(DATEADD(QUARTER, DATEDIFF(QUARTER, @BaseDate, D.DateRange), @BaseDate)), 
          1
      ) AS QuartersSort                                                      -- For quarter sorting
    
    -- === Offsets (differences from today) ===
    -- Useful for relative time filters like "last 30 days", "last month"
    , YEAR(D.DateRange) - YEAR(@Today) AS OffsetYear       -- Difference in years
    , DATEDIFF(MONTH, @Today, D.DateRange) AS OffsetMonth  -- Difference in months
    , DATEDIFF(DAY, @Today, D.DateRange) AS OffsetDay      -- Difference in days
    
    -- === Additional date attributes ===
    , CAST(D.DateRange AS DATE) AS DateLong                                 -- For long date formatting
    , CASE 
          WHEN EOMONTH(D.DateRange) = D.DateRange THEN CAST(1 AS BIT) 
          ELSE CAST(0 AS BIT) 
      END AS Is_EoM                                                         -- End of month?
    
    -- === Calendar weeks ===
    , 'CW ' + RIGHT('0' + CAST(DATEPART(ISO_WEEK, D.DateRange) AS VARCHAR(2)), 2) AS Calendar_Weeks  -- "CW 01" - "CW 52"
    , DATEPART(ISO_WEEK, D.DateRange) AS Calendar_Weeks_SORT                                       -- Numeric value for sorting
    
    -- === Weekdays ===
    -- Uses DATEPART(WEEKDAY) instead of CAST(date AS int), because the latter is not allowed
    , DATEFROMPARTS(1900, 1, DATEPART(WEEKDAY, D.DateRange)) AS WeekdayDate      -- For weekday names
    , DATEFROMPARTS(1900, 1, DATEPART(WEEKDAY, D.DateRange)) AS WeekdayDateLong  -- For long weekday names
    , DATEPART(WEEKDAY, D.DateRange) AS WeekDayNum                               -- Numeric weekday (depends on DATEFIRST)
    
    -- === Additional numeric attributes ===
    , DATEPART(DAYOFYEAR, D.DateRange) AS DayOfYear    -- Day of year (1-365/366)
    , MONTH(D.DateRange) AS MonthNumber                -- Month number (1-12)
    
    -- === Workday flags ===
    -- Monday (2) through Friday (6) = workday
    , CASE 
        WHEN DATEPART(WEEKDAY, D.DateRange) BETWEEN 2 AND 6 THEN 'Workday'
        ELSE 'Weekend'
      END AS Workday
    
    -- Monday (2) through Saturday (7) = business day
    , CASE 
        WHEN DATEPART(WEEKDAY, D.DateRange) BETWEEN 2 AND 7 THEN 'Business day'
        ELSE 'Rest day'
      END AS BusinessDay

FROM dim.DateRange AS D